# 03 — Audio Preprocessing for Wav2Vec2

This notebook prepares every audio file for Wav2Vec2 training.

Pipeline

Load Audio
→ Convert Stereo to Mono
→ Resample to 16 kHz
→ Peak Normalize
→ Store Processed Waveforms
→ Save Processed Dataset

## 1. Imports

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import librosa
import soundfile as sf

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from tqdm.auto import tqdm

np.random.seed(42)

TARGET_SAMPLE_RATE = 16000

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Load Dataframe

This assumes a prior notebook (e.g. `01_data_prep` / `02_visualization`) produced a
dataframe with at least:
- `filepath` — path to the audio file (.wav)
- `emotion` — label string (e.g. 'happy', 'sad', 'angry', ...)

Adjust the path/columns below to match your actual metadata file.


In [2]:
# ============================================================
# Load Dataset from Notebook 2
# ============================================================

from pathlib import Path
import pickle

BASE_DIR = Path.cwd()

# Go back to the SER project root
PROJECT_ROOT = BASE_DIR.parent.parent

OUTPUT_DIR = PROJECT_ROOT / "outputs"

INPUT_DATASET = OUTPUT_DIR / "ravdess_dataset.pkl"

print("Loading dataset from:")
print(INPUT_DATASET)

if not INPUT_DATASET.exists():
    raise FileNotFoundError(
        f"{INPUT_DATASET} not found. Run Notebook 2 first."
    )

with open(INPUT_DATASET, "rb") as f:
    df = pickle.load(f)

print("Dataset loaded successfully")

print("Emotion Classes:")
print(sorted(df["emotion"].unique()))

print("Number of classes:")
print(df["emotion"].nunique())
print("Loading dataset from:")
print(INPUT_DATASET)

if not INPUT_DATASET.exists():
    raise FileNotFoundError(
        f"\nDataset not found:\n{INPUT_DATASET}\n"
        "Run Notebook 2 first."
    )

with open(INPUT_DATASET, "rb") as f:
    df = pickle.load(f)

print("="*60)
print("Dataset Loaded Successfully")
print("="*60)

print("Rows :", len(df))
print("Columns :", len(df.columns))

display(df.head())

Loading dataset from:
/Users/devanshbansal/Desktop/ser/outputs/ravdess_dataset.pkl
Dataset loaded successfully
Emotion Classes:
['angry', 'calm', 'disgust', 'fearful', 'happy', 'sad', 'surprised']
Number of classes:
7
Loading dataset from:
/Users/devanshbansal/Desktop/ser/outputs/ravdess_dataset.pkl
Dataset Loaded Successfully
Rows : 1440
Columns : 12


,file_path,filename,emotion,emotion_code,actor,gender,intensity,statement,statement_text,repetition,vocal_channel,modality
0,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-01-01-01-01-01.wav,calm,01,1,male,normal,1,Kids are talking by the door,1,speech,audio_only
1,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-01-01-01-02-01.wav,calm,01,1,male,normal,1,Kids are talking by the door,2,speech,audio_only
2,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-01-01-02-01-01.wav,calm,01,1,male,normal,2,Dogs are sitting by the door,1,speech,audio_only
3,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-01-01-02-02-01.wav,calm,01,1,male,normal,2,Dogs are sitting by the door,2,speech,audio_only
4,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-02-01-01-01-01.wav,calm,02,1,male,normal,1,Kids are talking by the door,1,speech,audio_only


## 3. Audio Preprocessing Function
For each file:
1. Load audio at a fixed sample rate
2. Normalize amplitude (peak normalization)
3. Prepare raw waveform input for Wav2Vec2.

Steps:
1. Load audio
2. Convert stereo to mono
3. Resample to 16 kHz
4. Peak normalize
5. Convert to float32
   Spectral Centroid, Spectral Bandwidth, Spectral Contrast, Spectral Rolloff
4. Pool every frame-level feature into **mean + std** across time -> fixed-length vector


In [3]:
def preprocess_audio(filepath, target_sr=TARGET_SAMPLE_RATE):
    """
    Load an audio file and preprocess it for Wav2Vec2.

    Steps:
    1. Load original audio.
    2. Convert stereo to mono.
    3. Resample to target sample rate.
    4. Peak normalize.
    5. Convert to float32.
    """

    y, sr = librosa.load(
        filepath,
        sr=None,
        mono=False
    )

    # Stereo → Mono
    if y.ndim > 1:
        y = np.mean(y, axis=0)

    # Resample
    if sr != target_sr:
        y = librosa.resample(
            y,
            orig_sr=sr,
            target_sr=target_sr
        )

    # Peak normalization
    peak = np.max(np.abs(y))

    if peak > 0:
        y = y / peak

    # Handle completely silent files
    if len(y) == 0:
        y = np.zeros(target_sr, dtype=np.float32)

    return y.astype(np.float32), target_sr

## 4. Preprocess Audio Files

Run extraction across the full dataset. Failed files are logged and skipped (not silently dropped).

In [4]:
# ==========================================================
# Preprocess all audio files for Wav2Vec2
# ==========================================================

processed_audio = []
failed_files = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Preprocessing Audio"
):

    filepath = row["file_path"]

    try:
        y, sr = preprocess_audio(filepath)

        processed_audio.append({
            "waveform": y,
            "sample_rate": sr
        })

    except Exception as e:
        failed_files.append((filepath, str(e)))

        processed_audio.append({
            "waveform": None,
            "sample_rate": None
        })

print("=" * 60)
print(f"Successfully processed : {len(df) - len(failed_files)}")
print(f"Failed                 : {len(failed_files)}")
print("=" * 60)

if failed_files:
    print("\nFirst few failed files:")
    for fp, err in failed_files[:10]:
        print(fp)
        print(err)
        print("-" * 40)

Preprocessing Audio: 100%|██████████| 1440/1440 [00:04<00:00, 323.91it/s]

Successfully processed : 1440
Failed                 : 0


## 5. Build Processed Audio DataFrame

Combine the extracted feature matrix with metadata (filepath, emotion, etc.).

In [5]:
# ==========================================================
# Build Processed Audio DataFrame
# ==========================================================

processed_df = df.copy()

processed_df["waveform"] = [
    item["waveform"] for item in processed_audio
]

processed_df["sample_rate"] = [
    item["sample_rate"] for item in processed_audio
]

print("=" * 60)
print("Processed audio dataframe created")
print("=" * 60)

print(processed_df.shape)

display(processed_df.head())

Processed audio dataframe created
(1440, 14)


,file_path,filename,emotion,emotion_code,actor,gender,intensity,statement,statement_text,repetition,vocal_channel,modality,waveform,sample_rate
0,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-01-01-01-01-01.wav,calm,01,1,male,normal,1,Kids are talking by the door,1,speech,audio_only,"[3.509954e-06, -4.807788e-06, 6.262959e-06, -7...",16000
1,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-01-01-01-02-01.wav,calm,01,1,male,normal,1,Kids are talking by the door,2,speech,audio_only,"[-8.7073524e-05, -0.00017682035, 4.0303014e-05...",16000
2,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-01-01-02-01-01.wav,calm,01,1,male,normal,2,Dogs are sitting by the door,1,speech,audio_only,"[0.00024561264, 0.00046495523, 0.00055057195, ...",16000
3,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-01-01-02-02-01.wav,calm,01,1,male,normal,2,Dogs are sitting by the door,2,speech,audio_only,"[0.00038872915, 0.000337521, -1.9962981e-05, -...",16000
4,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-02-01-01-01-01.wav,calm,02,1,male,normal,1,Kids are talking by the door,1,speech,audio_only,"[0.0003747411, 1.66967e-05, -1.6409602e-05, 1....",16000


## 6. Verify Processed Audio

Drop rows that failed extraction entirely, and impute any stray NaNs (e.g. std=0 edge cases don't produce NaNs, but very short/silent clips might).

In [6]:
# ==========================================================
# Verify Processed Audio
# ==========================================================

print("=" * 60)
print("Processed Audio Verification")
print("=" * 60)

print(f"Total samples            : {len(processed_df)}")

valid_audio = processed_df["waveform"].notna().sum()

print(f"Successfully processed   : {valid_audio}")
print(f"Failed                   : {len(processed_df) - valid_audio}")

print(f"Target sample rate       : {TARGET_SAMPLE_RATE} Hz")

waveform_lengths = processed_df["waveform"].dropna().map(len)

print(f"Shortest clip            : {waveform_lengths.min()} samples")
print(f"Longest clip             : {waveform_lengths.max()} samples")
print(f"Average length           : {waveform_lengths.mean():.0f} samples")

print("\nUnique sample rates:")
print(processed_df["sample_rate"].value_counts())

assert processed_df["sample_rate"].dropna().eq(TARGET_SAMPLE_RATE).all(), \
    "Some files were not resampled correctly."

print("\nVerification Passed ✓")
print("=" * 60)

Processed Audio Verification
Total samples            : 1440
Successfully processed   : 1440
Failed                   : 0
Target sample rate       : 16000 Hz
Shortest clip            : 46981 samples
Longest clip             : 84351 samples
Average length           : 59211 samples

Unique sample rates:
sample_rate
16000    1440
Name: count, dtype: int64

Verification Passed ✓


In [7]:
# ==========================================================
# Verify Processed Dataset
# ==========================================================

print("=" * 70)
print("PROCESSED DATASET SUMMARY")
print("=" * 70)

print(f"Total samples           : {len(processed_df)}")
print(f"Successfully processed  : {processed_df['waveform'].notna().sum()}")
print(f"Failed                  : {processed_df['waveform'].isna().sum()}")

print(f"\nTarget sample rate      : {TARGET_SAMPLE_RATE} Hz")

print("\nUnique sample rates:")
print(processed_df["sample_rate"].value_counts(dropna=False))

waveform_lengths = processed_df["waveform"].dropna().map(len)

print("\nWaveform Statistics")
print(f"Shortest clip           : {waveform_lengths.min()} samples")
print(f"Longest clip            : {waveform_lengths.max()} samples")
print(f"Average length          : {waveform_lengths.mean():.0f} samples")

print("\nChecking normalization...")

max_peak = processed_df["waveform"].dropna().map(lambda x: np.max(np.abs(x))).max()

print(f"Maximum absolute amplitude : {max_peak:.6f}")

assert processed_df["sample_rate"].dropna().eq(TARGET_SAMPLE_RATE).all(), \
    "Some files were not resampled correctly."

assert max_peak <= 1.000001, \
    "Some waveforms are not properly normalized."

print("\n✓ All audio files are correctly resampled and normalized.")
print("=" * 70)

PROCESSED DATASET SUMMARY
Total samples           : 1440
Successfully processed  : 1440
Failed                  : 0

Target sample rate      : 16000 Hz

Unique sample rates:
sample_rate
16000    1440
Name: count, dtype: int64

Waveform Statistics
Shortest clip           : 46981 samples
Longest clip            : 84351 samples
Average length          : 59211 samples

Checking normalization...
Maximum absolute amplitude : 1.000000

✓ All audio files are correctly resampled and normalized.


## 7. Save Processed Dataset

Persist the processed waveforms and metadata for Wav2Vec2 training.

In [9]:
# ==========================================================
# Save Processed Dataset
# ==========================================================

print("\nEmotion Classes:")
print(sorted(processed_df["emotion"].unique()))

print(
    "Number of Classes:",
    processed_df["emotion"].nunique()
)
processed_df.drop(columns=["waveform"]).to_csv(
    OUTPUT_DIR / "processed_audio_metadata.csv",
    index=False
)

print("=" * 70)
print("Processed dataset saved successfully")
print("=" * 70)

print(f"Samples          : {len(processed_df)}")
print(f"Target SR        : {TARGET_SAMPLE_RATE} Hz")

print("\nSaved files:")
print(OUTPUT_DIR / "processed_audio_dataset_7classes.pkl")
print(OUTPUT_DIR / "processed_audio_metadata_7classes.csv")
assert processed_df["emotion"].nunique() == 7
with open(
    OUTPUT_DIR / "processed_audio_dataset.pkl",
    "wb"
) as f:
    pickle.dump(processed_df, f)


Emotion Classes:
['angry', 'calm', 'disgust', 'fearful', 'happy', 'sad', 'surprised']
Number of Classes: 7
Processed dataset saved successfully
Samples          : 1440
Target SR        : 16000 Hz

Saved files:
/Users/devanshbansal/Desktop/ser/outputs/processed_audio_dataset_7classes.pkl
/Users/devanshbansal/Desktop/ser/outputs/processed_audio_metadata_7classes.csv


## Summary

- Loaded every RAVDESS audio file.
- Converted stereo audio to mono.
- Resampled every file to **16 kHz**.
- Applied peak normalization.
- Verified all processed waveforms.
- Saved the processed dataset for Wav2Vec2 training.

Outputs:

- `processed_audio_dataset_7classes.pkl`
- `processed_audio_dataset_7classes.csv`

Next notebook:

**04_wav2vec2_training.ipynb**